# Introduction: ROOT objects, ROOT files, TTrees, and the road to RDataFrame

This short notebook is meant as a **prelude to the RDataFrame tutorial**, not as a full ROOT course.

By the end of this notebook, you should know:

- what ROOT is used for in physics analyses,
- what histograms and ROOT files are,
- what a `TTree` is and why it is columnar,


## What is ROOT?

ROOT is a scientific software framework widely used in high-energy physics and related fields.

In practice, analysts often use ROOT for three closely connected tasks:

1. storing large datasets in ROOT files,
2. reading columnar datasets stored as TTrees,
3. producing histograms and other analysis outputs.

- ROOT is written mainly in **C++**, with powerful **Python** bindings

- Adopted in High Energy Physics and other sciences (but also industry)
  - **1 EB** of data in ROOT format
  - Fits and parameters’ estimations for discoveries (e.g. the Higgs)
  - Thousands of ROOT plots in scientific publications
  
### ROOT's building blocks

ROOT can be seen as a collection of building blocks for various activities, such as:
- **Data analysis: histograms, graphs, functions**
- **High-level analysis interfaces**: RDataFrame
- **Statistical tools** (RooFit/RooStats): rich modeling and statistical inference
- **I/O**: storage of any C++ object, **column-wise** storage of datasets
- **Math**: non-trivial functions (e.g. Erf, Bessel), optimised math functions
- **C++ interpretation**: full language compliance
- **Multivariate analysis** (TMVA): e.g. Boosted decision trees, Neural Nets
- **Advanced graphics** (2D, 3D, event display)
- And more: HTTP servering,  JavaScript visualisation

# ROOT and Python

## A powerful Python interface

Most of the core parts of the ROOT software library are developed in C++ to provide efficient and easily scalable execution. Like many other scientific Python packages such as numpy, ROOT can be easily used in Python thanks to a set of language bindings (in this case, between C++ and Python).

## How does it work?

The ROOT Python interface relies on the [cppyy](https://cppyy.readthedocs.io) technology to provide dynamic and automatic bindings that do not require boilerplate code to bridge the two languages. This in turn relies on the C++ interpreter [cling](https://cling.readthedocs.io) to access reflection information and JIT compile the code needed to access the C++ functionalities. In practice, any C++ function or class that can be exposed via the C++ interpreter gets an automatic representation in Python without need for further development or user interaction (see examples below). On top of the automatic bindings, the ROOT project contains custom Python code to enhance C++ features with a Pythonic feeling. All such enhancements are collectively called "pythonizations".

## Using ROOT from Python

ROOT is written mainly in C++, but it provides Python bindings through `PyROOT`.

This means that from a Python notebook we can create and manipulate C++ ROOT objects such as histograms, files and TTrees.

In [ ]:
import ROOT

A good first check is to print the ROOT version used in the environment.

In [ ]:
print(ROOT.gROOT.GetVersion())

## Calling user-defined C++ code in Python

The user can write custom C++ code and access it in their Python application. For example, it is possible to declare a C++ function, as it is done below by passing its code as a string argument of the `Declare` function:

In [ ]:
ROOT.gInterpreter.Declare("""
double add(double a, double b) {
    return a + b;
}
""")

ROOT.add(3.14, 100)

In [ ]:
ROOT.gInterpreter.Declare("""
double divide(double a, double b){
    return a / b;
}
"""
)
ROOT.divide(100, 2.)

### What about code in C++ libraries?

ROOT also allows calling into code present outside of the same Python application, for example code written in external libraries. This enables you to write high-performance C++, compile it and use it from Python.

More information can be found [here](https://root.cern/manual/python/#loading-user-libraries-and-just-in-time-compilation-jitting).

## Type conversions

When calling C++ code from Python, there is a conversion between the Python arguments we pass and the C++ arguments that the C++ side expects. The ROOT Python interface takes care of such conversion automatically, for example from Python integer to C++ integer:

In [ ]:
ROOT.gInterpreter.Declare("void print_integer(int i) { std::cout << i << std::endl; }")

ROOT.print_integer(10)

# Checkpoint

Try to answer before moving on:

- Will ROOT.print_integer(2^33-1) work? Why ROOT.print_integer(2^31-1) works but ROOT.print_integer(2^31) does not work?

In [ ]:
ROOT.print_integer(2**31-1)

An example of a useful allowed conversion is Python list to `std::vector`:

In [ ]:
ROOT.gInterpreter.Declare("""
void print_vector(const std::vector<std::string> &v) {
    for (auto &&s : v) {
        std::cout << s << std::endl;
    }
}
""")

ROOT.print_vector(['One','Two', 'Three'])

## A final note on performance

Being able to call into C++ from Python does not guarantee that the performance of your Python script will always be the best, no matter what code you write!

In general, any heavy computation should be pushed to C++, e.g. encapsulating it in some C++ function that you call from Python or relying on libraries with fast C/C++ implementations (e.g. ROOT, NumPy).

In the context of high-energy physics, iterating over the collision events in a dataset is a common operation. Such iteration in Python can be slow for big datasets and should only be done during short exploratory work. Later in this course we will see how the event loop can be efficiently executed in C++, even from a Python script, with the help of ROOT's [RDataFrame](https://root.cern/doc/master/classROOT_1_1RDataFrame.html).

```python
# This can be slow!
for event in dataset:
    h.Fill(event.field)
```

# ROOT in Jupyter

ROOT can be used in Jupyter notebooks, both in Python and C++. In this course we will focus only on Python, but for people interested in ROOT C++ notebooks some examples can be found [here](https://swan-gallery.web.cern.ch/root_primer/).

There are some specificities and extra features available when running ROOT from a notebook, and that's what will be covered in this section!


## Shortcut for injecting C++ code in a notebook

In a notebook, we can use the `%%cpp` magic to declare C++ code to the interpreter. If `%%cpp` is present in a cell, its whole content is interpreted and executed as C++, and it has C++ syntax coloring!

In [ ]:
%%cpp
void print_integer_2(int i) {
   std::cout << i << std::endl;
}

The function we just defined in the previous (C++) cell can be now invoked from the next (Python) cell:

In [ ]:
ROOT.print_integer_2(7)

## ROOT histograms: the most common analysis output
[Histogram class documentation](https://root.cern.ch/doc/master/classTH1.html)

A histogram divides a variable into bins and counts how many entries fall into each bin.

In physics analyses, histograms are used everywhere: reconstructed mass, energy, transverse momentum, timing, detector response, event counts after selections, and many others.

Here is a simple one-dimensional histogram:

In [ ]:
h = ROOT.TH1D(
    "my_histo",                 # internal object name
    "My histogram",             # title
    50,                         # number of bins
    0.0,                        # lower edge
    100.0,                      # upper edge
)

The constructor arguments define the object name, title, number of bins and x-axis range.

Now we fill the histogram with toy values sampled from a Gaussian distribution.

In [ ]:
rng = ROOT.TRandom3(42)

for _ in range(100_000):
    value = rng.Gaus(50.0, 10.0)
    h.Fill(value)

[Drawing options documentation](https://root.cern.ch/doc/master/classTHistPainter.html)

In a Jupyter notebook, we usually draw ROOT objects on a `TCanvas`.

In a notebook, we want to use the `%jsroot on` magic and explicitly draw a `TCanvas`.

In [ ]:
%jsroot on
c = ROOT.TCanvas()
h.SetLineColor(ROOT.kBlue)
h.SetFillColor(ROOT.kRed)
h.GetXaxis().SetTitle("x axis")
h.GetYaxis().SetTitle("Number of entries")
h.SetLineWidth(2)
h.Draw() # draw the histogram on the canvas
c.Draw() # draw the canvas on the screen

To check the full documentation you can always refer to https://root.cern/doc/master (and then switch to the documentation for your particular ROOT version with the drop-down menu at the top of the page).

A histogram is also an object that stores useful summary information.

In [ ]:
print(f"Entries: {h.GetEntries():.0f}")
print(f"Mean: {h.GetMean():.3f}")
print(f"RMS: {h.GetRMS():.3f}")

### Checkpoint

Try to answer before moving on:

- What happens if you change the number of bins from `50` to `10`?
- What happens if you change the histogram range from `(0, 100)` to `(30, 70)`?
- Why could the choice of binning matter in a statistical analysis?

## ROOT functions
The type that represents an arbitrary one-dimensional mathematical function in ROOT is [TF1](https://root.cern.ch/doc/master/classTF1.html).<br>
Similarly, [TF2](https://root.cern.ch/doc/master/classTF2.html) and [TF3](https://root.cern.ch/doc/master/classTF3.html) represent 2-dimensional and 3-dimensional functions.

As an example, let's define and plot a simple function:

In [ ]:
f1 = ROOT.TF1("f1", "gaus(x)", xmin=-10, xmax=10)
f1.SetParameters(10, 3., 2.) # constant, mean, sigma

In [ ]:
%jsroot on
c = ROOT.TCanvas()
f1.Draw("surf1")
c.Draw()

In [ ]:
f2 = ROOT.TF2("f2", "gaus(x) * gaus(y)", xmin=-5, xmax=5, ymin=-5, ymax=5)
f2.SetParameters(10, 3., 2., 5, -3., 2.)

In [ ]:
c = ROOT.TCanvas()
f2.Draw("surf1") # to get a surface instead of the default contour plot
c.Draw()

## ROOT graphs

[TGraph](https://root.cern/doc/master/classTGraph.html) is a type useful for scatter plots.

Their drawing options are documented [here](https://root.cern/doc/master/classTGraphPainter.html).

Like for histograms, the aspect of `TGraph`s can be greatly customized, they can be fitted with custom functions, etc.

In [ ]:
import numpy as np
x = np.arange(-20, 21, dtype=float)
y = x*x
g = ROOT.TGraph(n=x.size, x=x, y=y)

c4 = ROOT.TCanvas()
g.SetMarkerStyle(7)
g.SetLineColor(ROOT.kBlue)
g.SetTitle("My graph")
g.Draw()
c4.Draw()

### Plot example: histogram stack

In HEP, we often plot stacked histograms, for example to show the
contributions of different processes. This can be done with [THStack](https://root.cern.ch/doc/master/classTHStack.html).

In [ ]:
f1 = ROOT.TF1("f1", "gaus", -4.0, 4.0)

histos = [ROOT.TH1D(f"h{i}", "x", 64, -4.0, 4.0) for i in range(3)]

hs = ROOT.THStack("hs","")
hs.SetTitle(";x;Events")

colors = [46, 30, 38]
        
for i in range(len(histos)):
    h = histos[i]
    f1.SetParameters(1.0*(i+1), 0.0 + i, 1.0 + 2*i)
    h.FillRandom("f1", 100000)
    h.SetFillColor(colors[i])
    hs.Add(h)

c6 = ROOT.TCanvas()
hs.Draw()
c6.Draw()

##  ROOT files: storing analysis objects

ROOT [TFile](https://root.cern/doc/master/classTFile.html) are binary files that can store ROOT objects such as histograms and TTrees. 

They can be transparently _compressed_ to reduce disk usage

This is how you create a `TFile`:

In [ ]:
f = ROOT.TFile("my_file.root", "RECREATE")

The most common file modes are:
<center><img src="data/tfile1.png"><center>

and how you close it (note that when `f` is destroyed, the file is closed automatically):

In [ ]:
f.Close()

For notebooks and scripts, the safest pattern is to use a Python context manager: `with ROOT.TFile(...) as f:`.

It will automatically run the necessary cleanup for you:

In [ ]:
with ROOT.TFile("my_file.root", "RECREATE") as f:
    h = ROOT.TH1D("my_histo", "Example histogram", 100, -4, 4)
    f.WriteObject(h, h.GetName())

The `"my_histo"` argument of the `TH1D` constructor is the name of the histogram, and it is also how it will be identified inside the file, we'll see that in a minute.

We can check that the file was created and inspect its content.

In [ ]:
%%bash
ls -l my_file.root

Now read the histogram back from the file.

In [ ]:
with ROOT.TFile("my_file.root", "READ") as f:
    f.ls()

We can also do:

In [ ]:
%%bash
rootls -l my_file.root

## The HEP dataset

High Energy Physics data is made of many statistically independent collision events. 

Laying data into an "event class", then serialise and write out `N` instances of the class into a file would be very inefficient. 

In ROOT, a dataset is organised columns that can store elements of any C++ type:
* fundamental types: `int`, `float`
* C++ standard collections: `std::vector`, `std::map`
* User created C++ classes

The ROOT dataset is represented by the `TTree` class and is often simply called a tree. Columns in the dataset are instances of the `TBranch` class (often referred to as "branches").

<center><img src="data/dataset.png"></center>

- A `TTree` dataset can be written to a `TFile` (just like any other C++ object). 

For example:

| event | energy | time | category |
|---:|---:|---:|---:|
| 0 | 42.1 | 12.4 | 1 |
| 1 | 55.3 | 10.8 | 0 |
| 2 | 48.7 | 11.9 | 1 |

The important feature is that TTrees are stored **column-wise**. If your analysis only needs `energy`, ROOT does not have to read all other branches from disk.

This is one reason why ROOT is efficient for large physics datasets.